# Plot Clusterbase vs exp_014 (Notebook)

Este notebook gera um dashboard Plotly por dataset, comparando:
- `clusterbase` (exp_022)
- `quadapt` (exp_014)
- `baseline` (exp_014)

Cada subplot é ordenado localmente pela mediana do erro (melhor → pior), no mesmo estilo que você pediu.


In [6]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ===== Configuração de caminhos =====
clusterbase_csv = Path("results/clusterbase_batches.csv")
quadapt_csv = Path("../../experiments/exp_014/results/quadapt.csv")
baseline_csv = Path("../../experiments/exp_014/results/baseline.csv")
output_html = Path("results/clusterbase_vs_exp014.html")
height_per_dataset = 320
title = "Comparação local por dataset: Clusterbase vs exp_014"


def _find_col(df, candidates, csv_name):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    raise ValueError(f"Não encontrei nenhuma coluna {list(candidates)} em {csv_name}. Colunas: {list(df.columns)}")


def load_clusterbase(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "status" in df.columns:
        df = df[df["status"] == "ok"].copy()

    dataset_col = _find_col(df, ["dataset"], path.name)
    error_col = _find_col(df, ["abs_error", "mae", "error", "erro"], path.name)

    out = pd.DataFrame(
        {
            "dataset": df[dataset_col].astype(str),
            "modelo": "clusterbase",
            "erro": pd.to_numeric(df[error_col], errors="coerce"),
        }
    ).dropna(subset=["erro"])
    return out


def load_generic_method(path: Path, fallback_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    dataset_col = _find_col(df, ["dataset", "dataset_name"], path.name)
    error_col = _find_col(df, ["erro", "error", "mae", "abs_error"], path.name)

    if any(c.lower() in {"modelo", "model", "method"} for c in df.columns):
        model_col = _find_col(df, ["modelo", "model", "method"], path.name)
        modelo = df[model_col].astype(str)
    else:
        modelo = pd.Series([fallback_name] * len(df), index=df.index)

    out = pd.DataFrame(
        {
            "dataset": df[dataset_col].astype(str),
            "modelo": modelo,
            "erro": pd.to_numeric(df[error_col], errors="coerce"),
        }
    ).dropna(subset=["erro"])
    return out


frames = []
if clusterbase_csv.exists():
    frames.append(load_clusterbase(clusterbase_csv))
else:
    print(f"[WARN] Arquivo ausente: {clusterbase_csv}")

if quadapt_csv.exists():
    frames.append(load_generic_method(quadapt_csv, fallback_name="quadapt"))
else:
    print(f"[WARN] Arquivo ausente: {quadapt_csv}")

if baseline_csv.exists():
    frames.append(load_generic_method(baseline_csv, fallback_name="baseline"))
else:
    print(f"[WARN] Arquivo ausente: {baseline_csv}")

if not frames:
    raise FileNotFoundError("Nenhum CSV foi encontrado. Ajuste os caminhos no início da célula.")

df = pd.concat(frames, ignore_index=True).dropna(subset=["dataset", "modelo", "erro"])
print("Registros totais:", len(df))
print("Datasets:", df["dataset"].nunique())
print("Modelos:", sorted(df["modelo"].unique()))

datasets = sorted(df["dataset"].unique())
n_datasets = len(datasets)

fig = make_subplots(
    rows=n_datasets,
    cols=1,
    subplot_titles=[f"<b>{ds}</b>" for ds in datasets],
    vertical_spacing=max(0.001, min(0.04, 0.5 / max(n_datasets, 1))),
)

for i, ds in enumerate(datasets, start=1):
    df_ds = df[df["dataset"] == ds].copy()
    ordem_local = df_ds.groupby("modelo")["erro"].median().sort_values().index.tolist()

    for modelo in ordem_local:
        df_mod = df_ds[df_ds["modelo"] == modelo]
        fig.add_trace(
            go.Box(
                y=df_mod["erro"],
                name=modelo,
                boxpoints="outliers",
                legendgroup=modelo,
                showlegend=(i == 1),
            ),
            row=i,
            col=1,
        )

fig.update_layout(
    height=max(600, n_datasets * height_per_dataset),
    template="plotly_white",
    title_text=title,
    margin=dict(t=100, b=50, l=50, r=50),
    showlegend=True,
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(title_text="Erro")

output_html.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(output_html)
print(f"HTML salvo em: {output_html}")

fig.show()



Registros totais: 136800
Datasets: 30
Modelos: ['A_Shape', 'B_Tail', 'Baseline', 'C_Divergence', 'D_QDeriv', 'DyS_Topsoe', 'QuaDapt_DyS', 'clusterbase']
HTML salvo em: results/clusterbase_vs_exp014.html
